# Capítulo 18 · Laboratório 18: intermercados e liquidez
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance e do FRED (Federal Reserve de St. Louis) na sua máquina e guarda uma cópia local (`dados_cap18.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório testa o "novo normal" de John Murphy (palestra de 2011) por épocas, mede o Brasil na teia (dólar, commodities, emergentes), testa a rotação setorial de Sam Stovall, compara o balanço do Fed com o S&P 500 e o Bitcoin e mede a correlação do Bitcoin com o Nasdaq ano a ano.

In [ ]:
!pip install -q yfinance

In [ ]:
import io
import logging
import os
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap18.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
cr = lambda x: ("+" if x >= 0 else "") + f"{x:.2f}".replace(".", ",")
TICKERS = {"spx": "^GSPC", "juro10": "^TNX", "dolar": "DX-Y.NYB", "commod": "DBC", "ouro": "GLD", "tlt": "TLT",
           "ibov": "^BVSP", "brl": "BRL=X", "eem": "EEM", "btc": "BTC-USD", "nasdaq": "^IXIC",
           "xle": "XLE", "xlp": "XLP", "xlv": "XLV", "xlu": "XLU"}


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    D = pd.read_pickle(CACHE)
else:
    bruto = yf.download(list(TICKERS.values()), start="1990-01-01", end=FIM, auto_adjust=True, progress=False)["Close"]
    confere({"os 15 ativos": bruto})
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=WALCL&cosd=2002-12-18&coed=2026-09-30"
    for tentativa in range(5):                                   # o servidor do FRED às vezes derruba a conexão
        try:
            texto = urllib.request.urlopen(url, timeout=60).read().decode()
            break
        except Exception:
            if tentativa == 4:
                raise RuntimeError("Não consegui baixar o balanço do Fed (FRED) depois de 5 tentativas. O site pode estar"
                                   " fora do ar. Espere alguns minutos e rode a célula de novo.")
    fed = pd.read_csv(io.StringIO(texto), index_col=0, parse_dates=True).iloc[:, 0]
    D = {"px": bruto.rename(columns={v: k for k, v in TICKERS.items()}), "fed": fed.loc[:FIM]}
    pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
PX = D["px"]
SEM = PX.resample("W-FRI").last()
R = np.log(SEM).diff()                                          # retornos semanais (log)
R["juro10"] = SEM["juro10"].diff()                              # no juro, a variação em pontos percentuais
print(f"Parte 1. {PX.shape[1]} séries semanais, de {PX.index[0]:%Y} a {PX.index[-1]:%d/%m/%Y}; balanço do Fed até"
      f" {D['fed'].index[-1]:%d/%m/%Y}.")

## Parte 2: o novo normal de Murphy

In [ ]:
EPOCAS = [("1990 a 1997", "1990", "1997"), ("1998 a 2007", "1998", "2007"), ("2008 a 2020", "2008", "2020"),
          ("2021 a 2026", "2021", "2026")]
PARES = [("spx", "juro10", "S&P 500 x juro de 10 anos"), ("spx", "tlt", "S&P 500 x títulos longos (TLT)"),
         ("dolar", "commod", "Dólar (DXY) x commodities (DBC)"), ("spx", "commod", "S&P 500 x commodities (DBC)"),
         ("spx", "dolar", "S&P 500 x dólar (DXY)"), ("tlt", "commod", "Títulos longos x commodities"),
         ("ouro", "spx", "Ouro (GLD) x S&P 500")]
TAB_EPOCAS = {}
print("\nParte 2. Correlação dos retornos semanais em cada época (n.d. = sem dados suficientes)")
print("   " + " " * 34 + "".join(f"{e:>13}" for e, _, _ in EPOCAS))
for a, b, nome in PARES:
    vals = []
    for _, i, f in EPOCAS:
        x = R.loc[i:f, [a, b]].dropna()
        vals.append(x[a].corr(x[b]) if len(x) > 100 else np.nan)
    TAB_EPOCAS[nome] = vals
    print(f"   {nome:<34}" + "".join(f"{('n.d.' if np.isnan(v) else cr(v)):>13}" for v in vals))
ROLL = R["spx"].rolling(52).corr(R["juro10"])                   # correlação móvel de um ano

In [ ]:
import matplotlib.pyplot as plt
ROLL.loc["1991":].plot(figsize=(8, 3.5), color="black")
plt.axhline(0, color="gray")
plt.title("S&P 500 x juro de 10 anos: correlação em 52 semanas")
plt.show()

## Parte 3: o Brasil na teia

In [ ]:
IBOV_USD = np.log(SEM["ibov"] / SEM["brl"]).diff()
TAB_BR = []
print("\nParte 3. Ibovespa contra o mundo (correlação semanal, 2008 a 2026)")
for nome, x, y in [("Ibovespa x dólar/real", R["ibov"], R["brl"]),
                   ("Ibovespa x commodities (DBC)", R["ibov"], R["commod"]),
                   ("Ibovespa em dólar x emergentes (EEM)", IBOV_USD, R["eem"]),
                   ("Ibovespa x S&P 500", R["ibov"], R["spx"]),
                   ("Dólar/real x dólar global (DXY)", R["brl"], R["dolar"])]:
    d = pd.concat([x, y], axis=1).loc["2008":].dropna()
    c = d.iloc[:, 0].corr(d.iloc[:, 1])
    TAB_BR.append((nome, c))
    print(f"   {nome:<38} {cr(c)}")

## Parte 4: rotação setorial

In [ ]:
DEFENS = np.exp(R[["xlp", "xlv", "xlu"]].mean(axis=1).cumsum())   # cesta de mesmo peso: consumo básico, saúde, utilidades
SPXI = np.exp(R["spx"].cumsum())
FORCA = (DEFENS / SPXI).pct_change(13)                             # defensivos contra o S&P 500 em 13 semanas
ENERGIA = (np.exp(R["xle"].cumsum()) / SPXI).pct_change(13)
FUT13 = SEM["spx"].shift(-13) / SEM["spx"] - 1
TAB4 = pd.DataFrame({"def": FORCA, "ene": ENERGIA, "f": FUT13}).loc["1999":].dropna()
TAB_ROT = []
print("\nParte 4. S&P 500 nas 13 semanas seguintes, conforme a liderança dos setores (desde 1999)")
for nome, cond in [("defensivos liderando e energia perdendo", (TAB4["def"] > 0) & (TAB4["ene"] < 0)),
                   ("defensivos perdendo", TAB4["def"] < 0), ("qualquer semana", TAB4["def"].notna())]:
    x = TAB4.loc[cond, "f"]
    TAB_ROT.append((nome, len(x), x.mean(), (x > 0).mean()))
    print(f"   {nome:<42} {len(x):>5} semanas | média {ps(x.mean())} | positivo em {pc((x > 0).mean())}")

## Parte 5: liquidez

In [ ]:
FED = D["fed"].resample("W-FRI").last().ffill()
CRESCE = FED.pct_change(52) > 0                                    # balanço maior que um ano antes
TAB_FED = []
print(f"\nParte 5. Balanço do Fed (FRED, WALCL) na semana de {FED.index[-1]:%d/%m/%Y}: US$ {nr(FED.iloc[-1] / 1e6)}"
      f" trilhões, {ps(FED.pct_change(52).iloc[-1])} em 12 meses; pico de US$ {nr(FED.max() / 1e6)} trilhões em"
      f" {FED.idxmax():%m/%Y}")
for ativo, inicio in [("spx", "2003"), ("btc", "2015")]:
    d = pd.DataFrame({"c": CRESCE, "r": R[ativo]}).loc[inicio:].dropna()
    d["c"] = d["c"].astype(bool)
    for nome, cond in [("balanço crescendo", d["c"]), ("balanço encolhendo", ~d["c"])]:
        x = d.loc[cond, "r"]
        anual = np.exp(x.mean() * 52) - 1
        TAB_FED.append((ativo, nome, len(x), anual))
        print(f"   {'S&P 500' if ativo == 'spx' else 'Bitcoin':<8} com {nome:<20} {len(x):>4} semanas | retorno anualizado"
              f" {ps(anual)}")

## Parte 6: Bitcoin e Nasdaq

In [ ]:
TAB_BTC = []
print("\nParte 6. Correlação dos retornos semanais entre Bitcoin e Nasdaq, ano a ano")
for ano in range(2015, 2027):
    x = R.loc[str(ano), ["btc", "nasdaq"]].dropna()
    if len(x) > 20:
        c = x["btc"].corr(x["nasdaq"])
        TAB_BTC.append((ano, c))
print("   " + " | ".join(f"{a}: {cr(c)}" for a, c in TAB_BTC))

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as células das Partes 1 e 2. Mostre por que `R` usa o logaritmo dos fechamentos semanais, por que `juro10` entra pela variação em pontos (`diff`), e não pelo logaritmo, e por que uma época com 100 semanas ou menos aparece como n.d." Confira com a Parte 2, que imprime a correlação de cada par em cada época (S&P 500 x juro de 10 anos: −0,42 de 1990 a 1997 e +0,37 de 2008 a 2020).
2. **Adaptar:** "Numa célula nova, depois da Parte 3, baixe [ATIVO] com `yf.download` (`end=FIM` e `auto_adjust=True`), passe para semanal com `resample('W-FRI').last()`, calcule o retorno com `np.log(...).diff()` e meça a correlação com `R['brl']` e com `R['commod']` desde 2008, como na Parte 3. Não mude a frequência semanal nem o período." Antes de olhar o ativo novo, confira se a Parte 3 continua imprimindo os números da Tabela 18.1 (Ibovespa x dólar/real: −0,51).

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, troque os retornos semanais por mensais. Alguma relação muda de sinal?
2. Na Parte 4, use 26 semanas em vez de 13. A rotação defensiva passa a avisar alguma coisa?
3. Na Parte 5, troque a comparação de 12 meses por 3 meses. O resultado do balanço do Fed muda?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.